In [1]:
from neural_networks.Recurrent import RNN, FusionNN
from dataLoader import *
import numpy as np
import torch

In [2]:
def resize_tensor(x,size = (7,7)):
    x = x.unsqueeze(0)
    x = torch.nn.functional.interpolate(x, size=size, mode='bilinear', align_corners=False)
    return x.squeeze(0)
    
def load_images(dirName, reverse = False):
    files_train = sorted(dirName.glob("*.npz"))
    train_data = []
    for file in enumerate(files_train):
        file_name = file[1]
        with np.load(file_name) as npz:
            arr = np.ma.MaskedArray(**npz)
            data = arr.data
            data = np.append(data, [arr.mask[0]], axis=0)
            x = torch.tensor(data, dtype=torch.float32)
            x = resize_tensor(x)
            train_data.append(x)
        
    max_width = max(data.shape[1] for data in train_data)
    max_height = max(data.shape[2] for data in train_data)
    max_size = max_width if max_width > max_height else max_height

    return [resize_tensor(data) for data in train_data]        

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

msi = load_images(MSI_SATELLITE_TRAIN_DIR)
hsi = load_images(HSI_SATELLITE_DIR, True)
airborne = load_images(HSI_AIRBORNE_DIR)
y = gt_train_df[column_names].values
y = scaler.fit_transform(y)
y_binned = np.digitize(y, bins=np.linspace(y.min(), y.max() + 1, 5))
y = torch.tensor(y, dtype=torch.float32)
device = "cuda" if torch.cuda.is_available else "cpu"

In [5]:
from torch.utils.data import Dataset

class MultiSourceDataset(Dataset):
    def __init__(self, msi, hsi, airborne, targets):
        self.msi = msi
        self.hsi = hsi
        self.airborne = airborne
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.msi[idx],
            self.hsi[idx],
            self.airborne[idx],
            self.targets[idx]
        )

In [5]:
from sklearn.model_selection import train_test_split

dataset = MultiSourceDataset(msi, hsi, airborne, y)

indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(indices, train_size=0.8)

from torch.utils.data import Subset

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

#Dimensionality reduction

model = RNN(neurons=256, hidden_size=2048).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()

early_stopping_treshold = 64
best_val_loss = float('inf')

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory
/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [17]:
early_stopping_counter = 0
# model.load_state_dict(torch.load("models/nn/RNN.pt"))
# early_stopping_before = best_val_loss
for epoch in range(1000):
    model.train()
    train_loss = 0.0

    for msi_x, hsi_x, air_x, y in train_loader:
        msi_x = msi_x.to(device)
        hsi_x = hsi_x.to(device)
        air_x = air_x.to(device)
        y = y.to(device)

        output = model([msi_x, hsi_x, air_x])
        loss = loss_fn(output, y)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for msi_x, hsi_x, air_x, y in val_loader:
            msi_x = msi_x.to(device)
            hsi_x = hsi_x.to(device)
            air_x = air_x.to(device)
            y = y.to(device)

            output = model([msi_x, hsi_x])

            loss = loss_fn(output, y)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "models/nn/RNN.pt")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_treshold:
            break

print(best_val_loss)

Epoch: 0, train loss: 0.9361 val loss: 0.7750
Epoch: 1, train loss: 0.8188 val loss: 0.7299
Epoch: 2, train loss: 0.7992 val loss: 0.7381
Epoch: 3, train loss: 0.8026 val loss: 0.7411
Epoch: 4, train loss: 0.7917 val loss: 0.7186
Epoch: 5, train loss: 0.7681 val loss: 0.7244
Epoch: 6, train loss: 0.7556 val loss: 0.6985
Epoch: 7, train loss: 0.7526 val loss: 0.7235
Epoch: 8, train loss: 0.7534 val loss: 0.7314
Epoch: 9, train loss: 0.7423 val loss: 0.7077
Epoch: 10, train loss: 0.7343 val loss: 0.7089
Epoch: 11, train loss: 0.7301 val loss: 0.7238
Epoch: 12, train loss: 0.7355 val loss: 0.7135
Epoch: 13, train loss: 0.7397 val loss: 0.7103
Epoch: 14, train loss: 0.7557 val loss: 0.7022
Epoch: 15, train loss: 0.7257 val loss: 0.7141
Epoch: 16, train loss: 0.7358 val loss: 0.7356
Epoch: 17, train loss: 0.7516 val loss: 0.7353
Epoch: 18, train loss: 0.7605 val loss: 0.7294
Epoch: 19, train loss: 0.7402 val loss: 0.7402
Epoch: 20, train loss: 0.7322 val loss: 0.7133
Epoch: 21, train loss: 

In [16]:
from pandas import DataFrame

files_test_msi = sorted(MSI_SATELLITE_TEST_DIR.glob("*.npz"))
test_data_msi = []
for file in enumerate(files_test_msi):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)
        test_data_msi.append(x)
        
max_width = max(data.shape[1] for data in test_data_msi)
max_height = max(data.shape[2] for data in test_data_msi)
max_size = max_width if max_width > max_height else max_height

test_data_msi = [resize_tensor(data) for data in test_data_msi]
test_data_msi = torch.stack(test_data_msi).to(device)

files_test_hsi = sorted(HSI_SATELLITE_TEST_DIR.glob("*.npz"))
test_data_hsi = []
for file in enumerate(files_test_hsi):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)
        test_data_hsi.append(x)
        
max_width = max(data.shape[1] for data in test_data_msi)
max_height = max(data.shape[2] for data in test_data_msi)
max_size = max_width if max_width > max_height else max_height

test_data_hsi = [resize_tensor(data) for data in test_data_hsi]
test_data_hsi = torch.stack(test_data_hsi).to(device)
with torch.no_grad():
    predictions = model([test_data_msi, test_data_hsi])

predictions = predictions.cpu().numpy()
predictions = scaler.inverse_transform(predictions)

submission = DataFrame(data = predictions, columns = column_names)
submission.to_csv("fusion.csv", index_label="sample_index")

# Alternative: Fusing CNNs to neural network without recurrent NN

In [14]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True) # any change
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

# 8192 12000
model = FusionNN(neurons=256, hidden_size=12000).to(device) #only increase

criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-6) # Fine
loss_fn = torch.nn.MSELoss()

# 42580369114875793
early_stopping_treshold = 64
best_val_loss = float('inf')

In [15]:
model.load_state_dict(torch.load("models/nn/FusionNN.pt"))

<All keys matched successfully>

In [13]:
early_stopping_before = best_val_loss
early_stopping_counter = 0

for epoch in range(5000):
    model.train()
    train_loss = 0.0

    for msi_x, hsi_x, air_x, y in train_loader:
        msi_x = msi_x.to(device)
        hsi_x = hsi_x.to(device)
        air_x = air_x.to(device)
        y = y.to(device)

        output = model([msi_x, hsi_x])
        loss = loss_fn(output, y)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for msi_x, hsi_x, air_x, y in val_loader:
            msi_x = msi_x.to(device)
            hsi_x = hsi_x.to(device)
            air_x = air_x.to(device)
            y = y.to(device)

            output = model([msi_x, hsi_x])

            loss = loss_fn(output, y)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "models/nn/FusionNN.pt")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_treshold:
            print(epoch, best_val_loss)
            break

Epoch: 0, train loss: 0.9920 val loss: 0.8920
Epoch: 1, train loss: 0.9435 val loss: 0.8442
Epoch: 2, train loss: 0.8893 val loss: 0.7908
Epoch: 3, train loss: 0.8279 val loss: 0.7314
Epoch: 4, train loss: 0.7635 val loss: 0.6702
Epoch: 5, train loss: 0.6988 val loss: 0.6157
Epoch: 6, train loss: 0.6438 val loss: 0.5727
Epoch: 7, train loss: 0.6040 val loss: 0.5396
Epoch: 8, train loss: 0.5713 val loss: 0.5132
Epoch: 9, train loss: 0.5480 val loss: 0.4953
Epoch: 10, train loss: 0.5338 val loss: 0.4823
Epoch: 11, train loss: 0.5202 val loss: 0.4741
Epoch: 12, train loss: 0.5090 val loss: 0.4706
Epoch: 13, train loss: 0.5049 val loss: 0.4644
Epoch: 14, train loss: 0.4971 val loss: 0.4640
Epoch: 15, train loss: 0.4962 val loss: 0.4586
Epoch: 16, train loss: 0.4888 val loss: 0.4565
Epoch: 17, train loss: 0.4874 val loss: 0.4552
Epoch: 18, train loss: 0.4832 val loss: 0.4513
Epoch: 19, train loss: 0.4798 val loss: 0.4516
Epoch: 20, train loss: 0.4769 val loss: 0.4496
Epoch: 21, train loss: 

# Alternative 2: RNN with bands as timestamps instead of sources of images

In [22]:
from neural_networks.Recurrent import BandRNN
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torch.utils.data import Subset

scaler = StandardScaler()
hsi = load_images(MSI_SATELLITE_TRAIN_DIR, True)
y = gt_train_df[column_names].values
y = scaler.fit_transform(y)
y_binned = np.digitize(y, bins=np.linspace(y.min(), y.max() + 1, 5))
y = torch.tensor(y, dtype=torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"

indices = list(range(len(hsi)))
dataset = BandDataset(hsi,y)
indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices,
    train_size=0.8,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

from torch.utils.data import Subset

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

model = BandRNN(rnn_hidden_size=128, hidden_size=4096).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()

early_stopping_treshold = 64
best_val_loss = float('inf')

In [23]:
from torch.utils.data import Dataset

class BandDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, idx):
        return (self.x[idx], self.y[idx])

In [24]:
early_stopping_counter = 0
for epoch in range(5000):
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = loss_fn(output, y)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device)

            output = model(x)

            loss = loss_fn(output, y)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch: {epoch}, train loss: {avg_train_loss:.4f} val loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stopping_counter = 0
        torch.save(model.state_dict(), "models/nn/bands_msi.pt")
    else:
        early_stopping_counter += 1
        if early_stopping_counter >= early_stopping_treshold:
            break

print(best_val_loss)

Epoch: 0, train loss: 1.0012 val loss: 1.0058
Epoch: 1, train loss: 1.0025 val loss: 1.0003
Epoch: 2, train loss: 0.9974 val loss: 1.0001
Epoch: 3, train loss: 0.9931 val loss: 0.9963
Epoch: 4, train loss: 0.9919 val loss: 0.9814
Epoch: 5, train loss: 0.9575 val loss: 0.9133
Epoch: 6, train loss: 0.9248 val loss: 0.9421
Epoch: 7, train loss: 0.9157 val loss: 0.8935
Epoch: 8, train loss: 0.8951 val loss: 0.8874
Epoch: 9, train loss: 0.8902 val loss: 0.8837
Epoch: 10, train loss: 0.8835 val loss: 0.8723
Epoch: 11, train loss: 0.8706 val loss: 0.8609
Epoch: 12, train loss: 0.8609 val loss: 0.8473
Epoch: 13, train loss: 0.8569 val loss: 0.8428
Epoch: 14, train loss: 0.8665 val loss: 0.8495
Epoch: 15, train loss: 0.8583 val loss: 0.8424
Epoch: 16, train loss: 0.8507 val loss: 0.8447
Epoch: 17, train loss: 0.8623 val loss: 0.8407
Epoch: 18, train loss: 0.8536 val loss: 0.8365
Epoch: 19, train loss: 0.8454 val loss: 0.8335
Epoch: 20, train loss: 0.8452 val loss: 0.8503
Epoch: 21, train loss: 

In [25]:
from pandas import DataFrame

files_test = sorted(MSI_SATELLITE_TEST_DIR.glob("*.npz"))
test_data = []
for file in enumerate(files_test):
    file_name = file[1]
    with np.load(file_name) as npz:
        arr = np.ma.MaskedArray(**npz)
        data = arr.data
        data = np.append(data, [arr.mask[0]], axis=0)
        x = torch.tensor(data, dtype=torch.float32)
        x = resize_tensor(x)
        test_data.append(x)
        
max_width = max(data.shape[1] for data in test_data)
max_height = max(data.shape[2] for data in test_data)
max_size = max_width if max_width > max_height else max_height

test_data = [resize_tensor(data) for data in test_data]
test_data = torch.stack(test_data).to(device)
with torch.no_grad():
    predictions = model(test_data)

predictions = predictions.cpu().numpy()
predictions = scaler.inverse_transform(predictions)

submission = DataFrame(data = predictions, columns = column_names)
submission.to_csv("bandMSIReverse.csv", index_label="sample_index")